# 10 · Contexto de la secuencia: atacar los latidos S en racha

**De dónde sale.** Con la v4 confirmada en SVDB, los errores que quedan tienen un culpable claro:

| Error de la v4 en SVDB | Cantidad | % |
|---|---|---|
| S→N | 6 055 | 41 % |
| N→V | 3 324 | 22 % |
| S→V | 2 425 | 16 % |
| N→S | 2 032 | 14 % |
| V→N | 780 | 5 % |

**El 57 % de los errores son latidos S que no se detectan.** Y al separarlos por si vienen solos o
en racha aparece el mecanismo:

| | Latidos | Se | RR previo / RR local |
|---|---|---|---|
| S aislados | 8 795 | 0.365 | **0.78** (se ven prematuros) |
| S en racha | 3 399 | **0.148** | **0.99** (no se ven prematuros) |

Dentro de una racha supraventricular, el "RR local" ya está formado por latidos rápidos, así que el
latido prematuro **deja de parecerlo**. La referencia está contaminada por lo que quiere detectar.

**Las features nuevas** (todas desde la señal, sin etiquetas):

- **Dos escalas de referencia:** una mediana móvil local (20 latidos) y otra larga (200). Contra la
  larga, una racha entera sigue viéndose rápida.
- **Detector de racha:** fracción de latidos recientes acortados y cuántos latidos pasaron desde el
  arranque (el arranque sí es prematuro).
- **Irregularidad local** (coeficiente de variación de los RR): separa fibrilación auricular de
  taquicardia regular.
- **Vecinos:** RR de ±2 latidos y correlación de morfología con el latido anterior y el siguiente.

**Decidido antes de correr nada:** mayor F1 macro de 3 clases en DS1 con folds congelados; si
empata dentro de 0.01, gana el de mayor F1 de S; y **se descarta cualquier variante que aumente los
V→N más de un 10 %** respecto de la v4 (166 → tope 183).

> Proyecto educativo y de investigación. No es un dispositivo médico.

In [1]:
import numpy as np
import pandas as pd

from ecg import config
from ecg.evaluate import summary
from ecg.features import build_features, sequence_features
from ecg.models.baseline import BaselineClassifier
from ecg.segment import load_split
from ecg.train import CLASSES_V3, cross_validate

pd.set_option("display.precision", 3)
ds1 = load_split("ds1")
keep = ds1["y"] != "F"
X, F = ds1["X"][keep], ds1["F"][keep]
y, groups = ds1["y"][keep], ds1["record"][keep]

previo = np.where(np.roll(groups, 1) == groups, np.roll(y, 1), "N"); previo[0] = "N"
siguiente = np.where(np.roll(groups, -1) == groups, np.roll(y, -1), "N"); siguiente[-1] = "N"
en_racha = (y == "S") & ((previo == "S") | (siguiente == "S"))
aislado = (y == "S") & ~en_racha
print(f"DS1: {int(aislado.sum())} latidos S aislados y {int(en_racha.sum())} en racha")

DS1: 417 latidos S aislados y 526 en racha


## 1. ¿Qué ve cada feature nueva?

In [2]:
ctx, nombres = sequence_features(X, F, groups)
grupo = np.where(y == "N", "N", np.where(y == "V", "V",
                 np.where(en_racha, "S en racha", "S aislado")))
pd.DataFrame(ctx, columns=nombres).assign(grupo=grupo).groupby("grupo").median().round(2).T

grupo,N,S aislado,S en racha,V
rr_sobre_mediana_movil,1.00,0.76,0.94,0.77
rr_sobre_referencia_larga,1.01,0.75,0.61,0.73
cv_rr_local,0.05,0.10,0.11,0.27
min_rr_reciente,0.91,0.71,0.56,0.66
frac_rr_cortos_recientes,0.00,0.10,0.60,0.30
latidos_desde_arranque_racha,20.00,0.00,9.00,0.00
rr_dos_atras,1.00,1.00,0.62,0.97
rr_dos_adelante,1.00,1.01,0.62,0.97
corr_latido_previo,0.99,0.94,0.87,0.35
corr_latido_siguiente,0.99,0.93,0.91,0.39


**Conclusión:** la referencia larga es la que hace el trabajo. Un latido S en racha tiene un
cociente cercano a 1 contra la referencia local (parece normal) pero claramente menor contra la
larga. El detector de racha lo confirma: la fracción de latidos recientes acortados sube y el
contador de "latidos desde el arranque" deja de estar en su tope.

## 2. Candidatos

In [3]:
CANDIDATOS = {
    "v4 (referencia)": ("rr_norm+morph+rel+wave", 100),
    "v5 · v4 + contexto": ("rr_norm+morph+rel+wave+ctx", 100),
    "contexto sin morfología relativa": ("rr_norm+morph+wave+ctx", None),
    "chico: relativas + contexto": ("rr_norm+rel+ctx", 100),
}
filas, pred = {}, {}
for nombre, (fs, bloque) in CANDIDATOS.items():
    Z, _ = build_features(X, F, fs, records=groups, template_block=bloque)
    cv = cross_validate(
        lambda: BaselineClassifier(kind="xgb", feature_set=fs, classes=list(CLASSES_V3),
                                   template_block=bloque),
        Z, y, groups, fold_map=config.DS1_FOLD_MAP, classes=list(CLASSES_V3))
    p = pred[nombre] = cv["pred"]
    s = summary(y, p, CLASSES_V3)
    filas[nombre] = {**{k: v for k, v in s.items() if v is not None},
                     "Se S aislados": float((p[aislado] == "S").mean()),
                     "Se S en racha": float((p[en_racha] == "S").mean()),
                     "V→N": int(((y == "V") & (p == "N")).sum()),
                     "errores": int((y != p).sum()), "features": Z.shape[1]}
tabla = pd.DataFrame(filas).T
tabla[["features", "macro_f1", "N_F1", "S_Se", "S_+P", "S_F1", "Se S aislados", "Se S en racha",
       "V_Se", "V_+P", "V→N", "errores"]].round(3)

,features,macro_f1,N_F1,S_Se,S_+P,S_F1,Se S aislados,Se S en racha,V_Se,V_+P,V→N,errores
v4 (referencia),48.0,0.725,0.975,0.371,0.267,0.311,0.487,0.279,0.953,0.832,166.0,2355.0
v5 · v4 + contexto,58.0,0.743,0.976,0.439,0.315,0.367,0.568,0.337,0.959,0.821,139.0,2265.0
contexto sin morfología relativa,51.0,0.682,0.956,0.393,0.432,0.412,0.511,0.300,0.933,0.534,229.0,4147.0
chico: relativas + contexto,21.0,0.743,0.975,0.480,0.358,0.410,0.691,0.314,0.880,0.809,430.0,2418.0


In [4]:
tope = int(1.1 * tabla.loc["v4 (referencia)", "V→N"])
admitidas = tabla[tabla["V→N"] <= tope]
print(f"tope de V→N admitido (v4 + 10 %): {tope}")
print("descartadas por el guardarraíl:", list(tabla.index.difference(admitidas.index)))
mejor = admitidas["macro_f1"].max()
empate = admitidas[admitidas["macro_f1"] >= mejor - 0.01]
print("dentro de 0.01 del mejor:", list(empate.index), "-> elegida por F1 de S:",
      empate["S_F1"].idxmax())

tope de V→N admitido (v4 + 10 %): 182
descartadas por el guardarraíl: ['chico: relativas + contexto', 'contexto sin morfología relativa']
dentro de 0.01 del mejor: ['v5 · v4 + contexto'] -> elegida por F1 de S: v5 · v4 + contexto


**Conclusiones:**

- **El contexto mejora S sin tocar V:** la Se de S pasa de 0.371 a 0.439 y su F1 de 0.311 a 0.367.
  Los S aislados suben de 0.487 a 0.568 y los que vienen en racha de 0.279 a 0.337.
- **Y encima mejora V:** los latidos ventriculares leídos como normales bajan de 166 a 139. Tiene
  sentido: la correlación con los latidos vecinos también ayuda a identificar un V aislado.
- **La variante chica (21 features) empataba en F1 macro y detectaba aún más S** (Se 0.480), pero
  **queda descartada por el guardarraíl declarado**: sube los V→N de 166 a 430. Sin la morfología
  absoluta, el modelo gana en S a costa del error más grave. Es exactamente el tipo de decisión que
  el criterio fijado de antemano existe para evitar.
- Quitar las features relativas al paciente (tercera fila) derrumba todo (0.682): el contexto
  **suma** a la morfología relativa, no la reemplaza.

## 3. Qué queda sin resolver

In [5]:
p5 = pred["v5 · v4 + contexto"]
print("Se de S en racha:", round(float((p5[en_racha] == "S").mean()), 3))
errores = pd.Series([f"{a}→{b}" for a, b in zip(y, p5) if a != b]).value_counts()
display(errores.head(6).to_frame("v5"))
por_registro = pd.DataFrame({
    "latidos S": pd.Series({int(r): int(((groups == r) & (y == "S")).sum())
                            for r in np.unique(groups)}),
    "S detectados v5": pd.Series({int(r): int(((groups == r) & (y == "S") & (p5 == "S")).sum())
                                  for r in np.unique(groups)}),
})
por_registro = por_registro[por_registro["latidos S"] >= 20]
por_registro["Se"] = (por_registro["S detectados v5"] / por_registro["latidos S"]).round(3)
por_registro.sort_values("latidos S", ascending=False)

Se de S en racha: 0.337


,v5
N→S,886
N→V,696
S→N,434
V→N,139
S→V,95
V→S,15


,latidos S,S detectados v5,Se
209,383,187,0.488
201,128,39,0.305
207,106,9,0.085
118,96,34,0.354
220,94,91,0.968
223,73,37,0.507
124,31,0,0.000


**Conclusiones:**

- Aun con el contexto, **los S en racha siguen siendo el caso difícil** (Se 0.337 contra 0.568 de
  los aislados). El contexto ayuda pero no resuelve: cuando la racha es larga, ni siquiera la
  referencia de 200 latidos se salva de la contaminación.
- El caso extremo ya lo medimos en SVDB: el registro 865, con el 58 % de sus latidos
  supraventriculares, tiene Se de S igual a 0.00 con cualquier versión. **Es el límite del enfoque,
  no un problema de estas features.**
- El camino que queda para S es el que no tomamos todavía: **la onda P**, que es la definición
  clínica de un latido supraventricular. Es más frágil de medir (en SVDB la señal viene a 128 Hz),
  y por eso quedó después de esta idea.

## Resumen

**v5 = v4 + contexto de la secuencia de latidos** (58 features).
`python -m ecg.train baseline-v5` → `models/baseline_v5.joblib`.

| | v4 | **v5** |
|---|---|---|
| F1 macro (3 clases) | 0.725 | **0.743** |
| Se de S | 0.371 | **0.439** |
| F1 de S | 0.311 | **0.367** |
| Se de S aislados | 0.487 | **0.568** |
| Se de S en racha | 0.279 | **0.337** |
| V no detectados (RISK-01) | 166 | **139** |
| Errores | 2 355 | **2 265** |

**Lo que se aprendió**

1. **El problema de S no era el clasificador, era la referencia.** Medir la prematuridad contra un
   promedio local contaminado por los propios latidos prematuros hacía invisible a la mitad de los
   S. Con dos escalas de referencia, una parte del problema se recupera.
2. **Es la tercera vez que el mismo patrón funciona:** v2 normalizó el ritmo por paciente, v4 la
   morfología, y v5 corrige la referencia temporal. Todas mejoran por elegir **contra qué** se
   compara, no por cambiar el modelo.
3. **El guardarraíl declarado de antemano hizo su trabajo:** la variante que más S detectaba
   habría triplicado el error más grave del análisis de riesgo.

⚠️ **v5 está validada solo en DS1.** DS2, INCART y SVDB están gastadas, y European ST-T no sirve
para confirmar S (tiene 28 latidos S en 5 registros contra 35 671 N). La confirmación de esta
mejora queda pendiente de una base adecuada.